In [ ]:
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd

from atlas.io import (
    StitchedZarrStore,
    extract_s_number,
    get_metadata_for_stitched_tif,
    get_valid_slice_folders,
)
from atlas.stitching import stitch_ATLAS_tiles


In [ ]:
#testing data path
series_folder = Path(
    "/path/to/section-set"
)
print(series_folder.exists())

In [ ]:
series_list, failed_folders = get_valid_slice_folders(series_folder)
# Assign initial storage indices in numeric ATLAS section order.
series_list = sorted(series_list, key=extract_s_number)

In [ ]:
print(f"\nFound {len(series_list)} series.")

for f in series_list:
    print(f)

In [ ]:
store = StitchedZarrStore.open(series_folder / "stitched_sections.zarr")

buffer_in_microns = 1
max_shift_in_pixels = 200

# Stitch each section and keep its image and accompanying records in one store.
for raw_data_folder in series_list:
    for mif_file in raw_data_folder.glob("*.ve-mif"):
        if not mif_file.is_file():
            continue
        acquisition_metadata = get_metadata_for_stitched_tif(mif_file)
        atlas_section_number = acquisition_metadata["atlas_section_number"]
        section_key = f"S_{atlas_section_number}"
        if (section_key in store.sections
                and store.sections[section_key].attrs.get("status") == "complete"):
            print(f"Skipping {section_key}: already complete in the Zarr store.")
            continue

        print(f"Stitching {section_key} from {mif_file}...")
        try:
            stitched_img, tile_df, transform_dict, stitching_metadata = stitch_ATLAS_tiles(
                mif_file,
                buffer_microns=buffer_in_microns,
                max_shift_pixels=max_shift_in_pixels,
            )
            store.write_section(
                atlas_section_number=atlas_section_number,
                image=np.flipud(stitched_img),
                acquisition_metadata=acquisition_metadata,
                stitching_metadata=stitching_metadata,
                transforms={key: value.tolist() for key, value in transform_dict.items()},
                source_folder=raw_data_folder,
                tile_dataframe=tile_df,
            )
        except Exception as error:
            print(f"Unexpected error stitching {section_key} from {mif_file}: {error}")


## Read a stitched section from the Zarr store

Use the ATLAS section number (the `S` number) to retrieve its image and stored records. The section metadata gives the corresponding Z index in the shared `(z, y, x)` image array.

In [ ]:
# Select an ATLAS section number, for example S_2.
atlas_section_number = 2
section_key = f"S_{atlas_section_number}"
section_record = dict(store.sections[section_key].attrs)

z_index = section_record["z_index"]
acquisition_metadata = section_record["acquisition_metadata"]
source_folder_name = acquisition_metadata["atlas_source_folder_name"]

# Read the image cropped to this section's original stitched shape.
section_image = store.read_section(atlas_section_number)
print(f"{section_key} is stored at z={z_index}")
print(f"Source folder name: {source_folder_name}")
print(f"Image shape: {section_image.shape}, dtype: {section_image.dtype}")

# Retrieve the other records stored with the section.
stitching_metadata = section_record["stitching_metadata"]
transforms = section_record["transforms"]

# Read the stored tile table as a DataFrame.
tile_table_csv = section_record["tile_table_csv"]
tile_df = pd.read_csv(StringIO(tile_table_csv)) if tile_table_csv else None

print("Acquisition metadata fields:", list(acquisition_metadata))
print("Stitching metadata fields:", list(stitching_metadata or {}))
print("Transform entries:", len(transforms or {}))
print("Tile table shape:", tile_df.shape if tile_df is not None else None)